# Day 16 — Forecasting Baselines

**Deliverables:** persistence and regression baselines, Random Forest, chronological holdout, MAE/RMSE, and prediction plots.

> **Data limitation:** The source water-potability dataset is static. The timestamps used here are synthetic. This experiment validates a forecasting workflow only; it does not establish real-world underwater sensor forecasting performance.

The target is **next-observation pH**. At each row, features available at that observation are used to predict the next row's pH.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
TARGET = "ph"

In [ ]:
# Load the synthetic timestamped dataset. In Colab, upload it if not found.
candidates = [
    Path("water_potability_temporal_synthetic.csv"),
    Path("/content/water_potability_temporal_synthetic.csv"),
    Path("/mnt/data/underwater_temporal_dataset/water_potability_temporal_synthetic.csv"),
]
data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csvs = [name for name in uploaded if name.lower().endswith(".csv")]
        if not csvs:
            raise FileNotFoundError("Upload water_potability_temporal_synthetic.csv")
        data_path = Path(csvs[0])
    except ImportError:
        raise FileNotFoundError("Place the synthetic timestamped CSV beside this notebook.")

df = pd.read_csv(data_path)
timestamp_col = next((c for c in ["timestamp","Timestamp","datetime","date_time","time"] if c in df.columns), None)
if timestamp_col is None:
    raise ValueError("No timestamp column found. Use the synthetic timestamped dataset.")
df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
df = df.dropna(subset=[timestamp_col]).sort_values(timestamp_col).reset_index(drop=True)
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
sensor_cols = [c for c in numeric_cols if c != "Potability"]
if TARGET not in sensor_cols:
    raise ValueError(f"Target {TARGET!r} is missing.")
print("Dataset:", data_path, "| rows:", len(df), "| target:", TARGET)

## Feature construction and prediction-time logic

- Current non-target sensor readings are used as predictors for the **next observed row**.
- Lag and rolling features use only the current and earlier observations.
- Calendar features use the timestamp at the prediction origin.
- Persistence uses the most recent available pH value, forward-filled from past observations only.
- No random shuffle is used. The final 15% is kept as chronological test data.

In [ ]:
work = df[[timestamp_col] + sensor_cols].copy()
work["target_next"] = work[TARGET].shift(-1)
work["prediction_time"] = work[timestamp_col].shift(-1)
work["persistence_prediction"] = work[TARGET].ffill()

feature_cols = [c for c in sensor_cols if c != TARGET]
for lag in [1, 2, 3, 6]:
    work[f"{TARGET}_lag_{lag}"] = work[TARGET].shift(lag - 1)
for window in [3, 6]:
    work[f"{TARGET}_rolling_mean_{window}"] = work[TARGET].rolling(window, min_periods=window).mean()
    work[f"{TARGET}_rolling_std_{window}"] = work[TARGET].rolling(window, min_periods=window).std()
feature_cols += [c for c in work.columns if c.startswith(f"{TARGET}_lag_") or c.startswith(f"{TARGET}_rolling_")]
work["hour_of_day"] = work[timestamp_col].dt.hour
work["day_of_week"] = work[timestamp_col].dt.dayofweek
feature_cols += ["hour_of_day", "day_of_week"]

model_df = work.dropna(subset=["target_next", "prediction_time", "persistence_prediction"]).copy()
print("Model rows:", len(model_df), "| feature count:", len(feature_cols))

In [ ]:
n = len(model_df)
train_end, val_end = int(n * 0.70), int(n * 0.85)
train = model_df.iloc[:train_end].copy()
validation = model_df.iloc[train_end:val_end].copy()
test = model_df.iloc[val_end:].copy()

for name, part in [("Train", train), ("Validation", validation), ("Test", test)]:
    print(f"{name}: {len(part)} rows | {part['prediction_time'].min()} to {part['prediction_time'].max()}")

X_train, y_train = train[feature_cols], train["target_next"]
X_test, y_test = test[feature_cols], test["target_next"]

## Models and metrics

The validation segment is reserved for model selection/tuning. The baseline run below fits on the training segment and reports final metrics on the untouched latest test segment. MAE and RMSE are reported in pH units.

In [ ]:
def evaluate(name, actual, predicted):
    return {"model": name,
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": float(np.sqrt(mean_squared_error(actual, predicted))),
            "test_rows": len(actual)}

persistence_pred = test["persistence_prediction"].to_numpy()

linear_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
linear_model.fit(X_train, y_train)
linear_pred = linear_model.predict(X_test)

rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                    random_state=RANDOM_STATE, n_jobs=-1))
])
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

metrics = pd.DataFrame([
    evaluate("Persistence", y_test, persistence_pred),
    evaluate("Linear Regression", y_test, linear_pred),
    evaluate("Random Forest", y_test, rf_pred)
]).sort_values("MAE")
display(metrics)

In [ ]:
predictions = pd.DataFrame({
    "prediction_time": test["prediction_time"].to_numpy(),
    "actual": y_test.to_numpy(),
    "persistence": persistence_pred,
    "linear_regression": linear_pred,
    "random_forest": rf_pred
}).sort_values("prediction_time")

plt.figure(figsize=(13, 5))
plt.plot(predictions["prediction_time"], predictions["actual"], label="Actual", linewidth=2)
for col, label in [("persistence","Persistence"), ("linear_regression","Linear Regression"), ("random_forest","Random Forest")]:
    plt.plot(predictions["prediction_time"], predictions[col], label=label)
plt.title(f"Actual vs Predicted {TARGET} — Chronological Test Set")
plt.xlabel("Prediction time (synthetic)")
plt.ylabel(TARGET)
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (col, label) in zip(axes, [("persistence","Persistence"), ("linear_regression","Linear Regression"), ("random_forest","Random Forest")]):
    ax.scatter(predictions["actual"], predictions[col], alpha=0.6)
    lo = min(predictions["actual"].min(), predictions[col].min())
    hi = max(predictions["actual"].max(), predictions[col].max())
    ax.plot([lo, hi], [lo, hi], linestyle="--")
    ax.set_title(label); ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
plt.suptitle(f"Predicted vs Actual — {TARGET}")
plt.tight_layout()
plt.show()

In [ ]:
# Export reproducible evidence files
out_dir = Path("day16_outputs")
out_dir.mkdir(exist_ok=True)
metrics.to_csv(out_dir / "forecast_metrics.csv", index=False)
predictions.to_csv(out_dir / "forecast_predictions.csv", index=False)
split = pd.DataFrame([
    {"split":"train","rows":len(train),"start":str(train["prediction_time"].min()),"end":str(train["prediction_time"].max())},
    {"split":"validation","rows":len(validation),"start":str(validation["prediction_time"].min()),"end":str(validation["prediction_time"].max())},
    {"split":"test","rows":len(test),"start":str(test["prediction_time"].min()),"end":str(test["prediction_time"].max())}
])
split.to_csv(out_dir / "chronological_split.csv", index=False)
print("Saved:", sorted(p.name for p in out_dir.iterdir()))

## Interpretation and limitations

- Compare the three MAE/RMSE values; lower is better.
- Check whether either learned model improves on persistence.
- Inspect periods with large prediction errors.
- **Do not interpret these metrics as real sensor forecasting performance:** timestamps were synthetically generated and the original dataset has no true temporal ordering.
- Before deployment or scientific claims, repeat this experiment using genuine time-stamped underwater sensor measurements.